# Import data

In [29]:
import pandas as pd
import numpy as np
# transaction file
df = pd.read_csv("../data/transactions_train.csv",dtype={
    'article_id': str
})
df.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687004,0.016932,2


In [30]:
df.dtypes

t_dat                object
customer_id          object
article_id           object
price               float64
sales_channel_id      int64
dtype: object

In [31]:
# Customer, Articles and Sample Submissions
df_articles = pd.read_csv("../data/articles.csv", dtype={
    'article_id': str, 
    'product_code': str
})

# 2. Customers: Usually fine as default, but good to keep clean
df_customers = pd.read_csv("../data/customers.csv", dtype={
    'customer_id': str
})

# 3. Sample Submission: Crucial to keep customer_id as string
df_sample_subs = pd.read_csv("../data/sample_submission.csv", dtype={
    'customer_id': str
})

## merging the data

In [32]:
df1 = pd.merge(df,df_customers,on = 'customer_id',how = 'left')
df2 = pd.merge(df1,df_articles,on = 'article_id',how = 'left')
df2.head(1)

,t_dat,customer_id,article_id,price,sales_channel_id,FN,Active,club_member_status,fashion_news_frequency,age,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2,NaN,NaN,ACTIVE,NONE,24.0,...,Expressive Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Lace push-up body with underwired, moulded, pa..."


## Extracting week numbers

In [33]:
df2['t_dat'] = pd.to_datetime(df2['t_dat'])
df2['week'] = df2['t_dat'].dt.strftime('%Y-%U')
df2.head(2)

,t_dat,customer_id,article_id,price,sales_channel_id,FN,Active,club_member_status,fashion_news_frequency,age,...,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,week
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2,NaN,NaN,ACTIVE,NONE,24.0,...,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Lace push-up body with underwired, moulded, pa...",2018-37
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.030492,2,NaN,NaN,ACTIVE,NONE,24.0,...,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Lace push-up bras with underwired, moulded, pa...",2018-37


In [34]:
df2.week.unique()

array(['2018-37', '2018-38', '2018-39', '2018-40', '2018-41', '2018-42',
       '2018-43', '2018-44', '2018-45', '2018-46', '2018-47', '2018-48',
       '2018-49', '2018-50', '2018-51', '2018-52', '2019-00', '2019-01',
       '2019-02', '2019-03', '2019-04', '2019-05', '2019-06', '2019-07',
       '2019-08', '2019-09', '2019-10', '2019-11', '2019-12', '2019-13',
       '2019-14', '2019-15', '2019-16', '2019-17', '2019-18', '2019-19',
       '2019-20', '2019-21', '2019-22', '2019-23', '2019-24', '2019-25',
       '2019-26', '2019-27', '2019-28', '2019-29', '2019-30', '2019-31',
       '2019-32', '2019-33', '2019-34', '2019-35', '2019-36', '2019-37',
       '2019-38', '2019-39', '2019-40', '2019-41', '2019-42', '2019-43',
       '2019-44', '2019-45', '2019-46', '2019-47', '2019-48', '2019-49',
       '2019-50', '2019-51', '2019-52', '2020-00', '2020-01', '2020-02',
       '2020-03', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08',
       '2020-09', '2020-10', '2020-11', '2020-12', 

In [35]:
# train test split
weeks = df2['week'].to_numpy()
is_test_week = (weeks == '2020-38')
df_testing = df2[is_test_week]
df_train = df2[~(is_test_week)]
print(df_testing.shape)
print(df_train.shape)

(96485, 36)
(31691839, 36)


In [36]:
df_testing.customer_id.nunique()

28845

## Creating the validation score function
## The Objective
Write a Python function `calculate_map12(actual_df, predicted_df)` that calculates the **Mean Average Precision at 12 (MAP@12)** for the validation set. This function will serve as the ultimate judge for every recommendation strategy we test.

The formula for Average Precision at $K$ is:
$$AP@K = \frac{1}{\min(m, K)} \sum_{k=1}^{K} P(k) \times rel(k)$$
Where $m$ is the number of actual relevant items, $P(k)$ is the precision at cutoff $k$, and $rel(k)$ is an indicator function (1 if the item at rank $k$ is relevant, 0 otherwise).

## The Inputs
1. **`actual_df`**: A Pandas DataFrame representing the ground truth (e.g., Week 38). Contains `customer_id` and `article_id` (the items they *actually* bought).
2. **`predicted_df`**: A Pandas DataFrame containing your predictions. Contains `customer_id` and a `prediction` column (a space-separated string of up to 12 predicted article IDs).

## The Rules
* **Rank Matters:** Hits at the top of the list (positions 1, 2, 3) contribute more to the score than hits at the bottom (10, 11, 12).
* **No Purchases = Ignored:** Customers who did not buy anything during the test period must be excluded from the final mean calculation entirely.
* **Output:** Return a single float representing the mean of all individual valid customer AP@12 scores.

---

In [37]:
def calculate_precision(actual,predictions,k = 12):
    actuals_arr = actual.split()
    predictions_arr = predictions.split()
    mean = 0 
    hit = 0
    for idx,pred in enumerate(predictions_arr):
        if pred in actuals_arr:
            hit = hit + 1
            mean1 = hit / (idx + 1)
            mean = mean + mean1
    return(mean/min(k,len(actuals_arr)))


In [38]:
# Test 1: The Perfect Prediction
# Model guesses the exact 3 items right at the top.
act1 = "A B C"
pred1 = "A B C X Y Z"
print("Test 1 (Perfect):", calculate_precision(act1, pred1))  
# EXPECTED: 1.0

# Test 2: The "Late Bloomer"
# Model gets them right, but wastes the first two slots with bad guesses.
act2 = "A B"
pred2 = "X Y A B"
print("Test 2 (Late Hits):", calculate_precision(act2, pred2))  
# EXPECTED: 0.416666... (Penalized for ranking them lower)

# Test 3: The "Partial Basket"
# Customer bought 5 things, model only found 1.
act3 = "A B C D E"
pred3 = "A X Y Z W"
print("Test 3 (Missed Items):", calculate_precision(act3, pred3))  
# EXPECTED: 0.2 (1.0 hit score divided by the 5 things they actually bought)

# Test 4: The Duplicate Trap
# Model accidentally predicts "A" twice. 
act4 = "A B"
pred4 = "A A B"
print("Test 4 (Duplicates):", calculate_precision(act4, pred4))
# EXPECTED: If you added the `seen` set, it should be 1.0. 
# If you didn't, it will glitch and output 1.5!

Test 1 (Perfect): 1.0
Test 2 (Late Hits): 0.41666666666666663
Test 3 (Missed Items): 0.2
Test 4 (Duplicates): 1.5


## Naive Baseline 
last 3 weeks top 12 items

In [39]:
df_train.head(1)

,t_dat,customer_id,article_id,price,sales_channel_id,FN,Active,club_member_status,fashion_news_frequency,age,...,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,week
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2,NaN,NaN,ACTIVE,NONE,24.0,...,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Lace push-up body with underwired, moulded, pa...",2018-37


In [40]:
# last 3 weeks 
df_tr_naive = df_train[df_train['week'].isin(['2020-35','2020-36','2020-37'])]
df_tr_naive.shape

(775754, 36)

In [41]:
top12 = df_tr_naive.groupby('article_id').size().reset_index(name = 'purchases').sort_values(by = 'purchases',ascending = False).head(12)
top_12_list = top12['article_id'].astype(str).to_list()

# 3. Join them into your space-separated Kaggle prediction string
baseline_prediction_string = " ".join(top_12_list)

print(baseline_prediction_string)

0751471001 0915526001 0909370001 0751471043 0915529003 0918292001 0706016001 0448509014 0918522001 0898694001 0924243001 0863595006


In [47]:
df_test_naive = df_testing.groupby('customer_id')['article_id'].apply(" ".join).reset_index(name='actual')
df_test_naive.head()

,customer_id,actual
0,00039306476aaf41a07fed942884f16b30abfa83a2a8be...,0624486001
1,0003e867a930d0d6842f923d6ba7c9b77aba33fe2a0fbf...,0827487003
2,000493dd9fc463df1acc2081450c9e75ef8e87d5dd17ed...,0757926001 0788575004 0640021019
3,000525e3fe01600d717da8423643a8303390a055c578ed...,0874110016
4,00077dbd5c4a4991e092e63893ccf29294a9d5c46e8501...,0158340001 0867966009 0915529003 0932798002 09...


In [48]:
df_test_naive['precision'] = df_test_naive.apply(
    lambda row: calculate_precision(row['actual'], baseline_prediction_string), 
    axis=1
)
df_test_naive.head()

,customer_id,actual,precision
0,00039306476aaf41a07fed942884f16b30abfa83a2a8be...,0624486001,0.0000
1,0003e867a930d0d6842f923d6ba7c9b77aba33fe2a0fbf...,0827487003,0.0000
2,000493dd9fc463df1acc2081450c9e75ef8e87d5dd17ed...,0757926001 0788575004 0640021019,0.0000
3,000525e3fe01600d717da8423643a8303390a055c578ed...,0874110016,0.0000
4,00077dbd5c4a4991e092e63893ccf29294a9d5c46e8501...,0158340001 0867966009 0915529003 0932798002 09...,0.0375


In [73]:
# avg_precision
f"Average Precision Score: {df_test_naive.precision.mean()*100}%"

'Average Precision Score: 0.6012882386217269%'

### Segment Aware Baseline
1) Check which segment the customer buys
2) Show top 12 per segment based on customer

In [62]:
# The correct way to name and sort your combined groups
df_counts = df_tr_naive.groupby(['index_group_name', 'article_id']).size().reset_index(name='counts')
df_counts = df_counts.sort_values(by=['index_group_name', 'counts'], ascending=[True, False])

df_counts.head()

,index_group_name,article_id,counts
30,Baby/Children,0309864002,96
3001,Baby/Children,0866482006,94
2754,Baby/Children,0860172002,77
3942,Baby/Children,0889574009,73
3007,Baby/Children,0866503003,69


In [64]:
segment_bestsellers = {}

for segment in df_counts['index_group_name'].unique():
    segment_df = df_counts[df_counts['index_group_name'] == segment]
    
    # Grab the top 12 article_ids
    top_12_ids = segment_df['article_id'].head(12).tolist()
    
    # Join into space-separated string
    segment_bestsellers[segment] = " ".join(top_12_ids)

import pprint
pprint.pprint(segment_bestsellers)

{'Baby/Children': '0309864002 0866482006 0860172002 0889574009 0866503003 '
                  '0309864013 0885919006 0885919009 0870525005 0852785003 '
                  '0866482011 0556560001',
 'Divided': '0706016001 0448509014 0715624001 0919365008 0714790020 0797988002 '
            '0867969003 0914441004 0706016003 0894780001 0706016002 0809238001',
 'Ladieswear': '0751471001 0915526001 0909370001 0751471043 0915529003 '
               '0918522001 0898694001 0924243001 0863595006 0865799006 '
               '0850917001 0896152002',
 'Menswear': '0685814001 0685814003 0685813001 0685813037 0685816002 '
             '0685814033 0685813043 0685814022 0685814005 0685813042 '
             '0685816001 0685814062',
 'Sport': '0918292001 0852584001 0905957001 0866731001 0918292004 0912204001 '
          '0730683050 0907527001 0905518001 0804992014 0924605001 0805000001'}


In [65]:
# segmenting customers per their choices
df_tr_naive.head(1)

,t_dat,customer_id,article_id,price,sales_channel_id,FN,Active,club_member_status,fashion_news_frequency,age,...,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,week
30916085,2020-08-30,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,0562245099,0.033881,2,1.0,1.0,ACTIVE,Regularly,56.0,...,A,Ladieswear,1,Ladieswear,15,Womens Everyday Collection,1009,Trousers,"5-pocket jeans in washed, superstretch denim w...",2020-35


In [67]:
#Creating user segments
user_segment_counts = df2.groupby(['customer_id', 'index_group_name']).size().reset_index(name='purchase_count')
user_segment_counts = user_segment_counts.sort_values(by=['customer_id', 'purchase_count'], ascending=[True, False])
user_features = user_segment_counts.drop_duplicates(subset=['customer_id'], keep='first')
user_features.head()

,customer_id,index_group_name,purchase_count
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,Ladieswear,13
6,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,Ladieswear,62
9,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,Ladieswear,12
12,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,Sport,2
13,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,Ladieswear,11


In [69]:
# Create a fast lookup dictionary: {customer_id: favorite_segment}
user_to_segment_dict = dict(zip(user_features['customer_id'], user_features['index_group_name']))

# Map predictions to your test dataframe
df_test_naive['segment_prediction'] = df_test_naive['customer_id'].map(user_to_segment_dict).map(segment_bestsellers)
df_test_naive.head()

,customer_id,actual,precision,segment_prediction
0,00039306476aaf41a07fed942884f16b30abfa83a2a8be...,0624486001,0.0000,0751471001 0915526001 0909370001 0751471043 09...
1,0003e867a930d0d6842f923d6ba7c9b77aba33fe2a0fbf...,0827487003,0.0000,0706016001 0448509014 0715624001 0919365008 07...
2,000493dd9fc463df1acc2081450c9e75ef8e87d5dd17ed...,0757926001 0788575004 0640021019,0.0000,0706016001 0448509014 0715624001 0919365008 07...
3,000525e3fe01600d717da8423643a8303390a055c578ed...,0874110016,0.0000,0706016001 0448509014 0715624001 0919365008 07...
4,00077dbd5c4a4991e092e63893ccf29294a9d5c46e8501...,0158340001 0867966009 0915529003 0932798002 09...,0.0375,0751471001 0915526001 0909370001 0751471043 09...


In [74]:
df_test_naive['precision_per_seg'] = df_test_naive.apply(
    lambda row: calculate_precision(row['actual'], row['segment_prediction']), 
    axis=1
)

# Check out the final mean score
f"Segment precision score : {df_test_naive['precision_per_seg'].mean() *100}%"

'Segment precision score : 0.8283478629703489%'

# H&M Recommender — Progress & Next Steps

## ✅ Done so far

### Phase 1: Evaluation engine
- Built custom `calculate_precision` for **MAP@12** on raw space-separated strings.
- **Fixed Index Bug:** replaced `.index()` with `enumerate()` so duplicate predictions don't get artificial bonus points.
- **Fixed Denominator Bug:** divide by `min(k, len(actuals_arr))` instead of `hit`, so the model is properly penalized for missing actual purchases.
- **Fixed leading-zero trap:** set `dtype={'article_id': str}` on load — `article_id` locked as clean 10-char strings.

### Phase 2: Baselines (time-based split, hold out final 7 days, MAP@12)
| Strategy | Description | MAP@12 |
|---|---|---|
| Global Popularity | Top 12 most-bought items overall → same list to everyone | **0.00601** |
| Segment-Aware (3wk) | Top 12 bestsellers per `index_group_name`, mapped by each customer's favourite segment | **0.00800** |

Segmentation gave **+33%** — confirms the EDA finding that customers are segment-loyal.

---

## 🎯 Next steps

### Phase 2.5 — Squeeze the heuristics (before ML)
- [ ] **Window sweep** — recompute per-segment bestsellers over 1 / 2 / 3 week windows, let validation pick. *(Hypothesis: 1wk > 3wk — fast fashion, hold-out is final 7 days.)*
- [ ] **Repurchase source** — add each customer's own recent purchases to their rec list (cheap points).

### Phase 3 — Candidate generation (the real ML prerequisite)
- [ ] **Build explicit candidate stage** — produce a pool of ~50–100 articles/customer, not a final 12. Union of: recent segment bestsellers + own repurchases + age/segment cold-start fallback. Then rank pool → top 12.
- [ ] **Measure candidate recall** — % of actual hold-out purchases present in the pool. This is the CEILING on MAP. Fix candidate gen first if low.
- [ ] **Audit validation for leakage** — every feature for week N uses only data strictly before week N.

### Phase 4 — Into ML (only after recall is solid)
- [ ] **Hand-build features** — segment-buy count, days since last purchase, article sales rank last week, age match.
- [ ] **Skim learning-to-rank vs classification** — why `LGBMRanker` / `lambdarank` exist (order matters, not probabilities).
- [ ] **Train LightGBM ranker** — input: candidate pool + features; target: purchased in hold-out week.

### Deferred (after a working ranker)
- [ ] Image-embedding visual similarity (high effort, marginal MAP)
- [ ] Neural / two-tower
- [ ] Engagement + channel as ranking features

---
**Note:** Slow path = fast path. Phase 3 is the real prerequisite; the rest can wait. The heuristics *are* the system — ML just re-weights signals you've already found.